# romacoustics — BRAS CR2 Validation

Validates the Laplace-domain ROM solver against measured impulse responses from the BRAS round-robin benchmark (Scene 9 / CR2 seminar room, 8.4 × 6.7 × 3.0 m).

**Reference:** Aspöck et al., Building and Room Acoustics Simulation (BRAS) benchmark, TU Berlin.

**Tests performed:**
1. FOM eigenfrequency validation (analytical vs computed)
2. Box room T30 at 125 and 250 Hz vs BRAS measured RIR
3. Arbitrary geometry (convex hull of BRAS OBJ) eigenfrequency cross-check
4. ROM accuracy: reduced basis vs full-order solution

In [ ]:
# Setup
import subprocess, sys, os

# Install romacoustics
if not os.path.exists('romacoustics'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Burhanuddin98/Reduced-Order-Modelling-SL.git',
                    'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'repo/romacoustics'],
                   check=True)
    # Also install gmsh for arbitrary geometry
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'gmsh'], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from romacoustics import Room
print(f'romacoustics imported OK')

## Test 1: Eigenfrequency Validation (2D rigid rectangle)

A 2D rectangular room with rigid walls has analytical eigenfrequencies:

$$f_{n,m} = \frac{c}{2} \sqrt{\left(\frac{n}{L_x}\right)^2 + \left(\frac{m}{L_y}\right)^2}$$

We compute the impulse response, take the FFT, and check that spectral peaks align with the analytical values.

In [ ]:
# 2D rigid rectangle: 2m x 1m
Lx, Ly = 2.0, 1.0
c = 343.0

room = Room.box_2d(Lx, Ly, ne=20, order=4)
room.set_source(0.7, 0.3, sigma=0.15)
room.set_receiver(1.5, 0.7)
room.set_boundary_fi(Zs=1e8)  # near-rigid

ir = room.solve(t_max=0.2, f_max=1000, Ns=500)
print(f'DOFs: {room.N}, IR samples: {len(ir.signal)}')

In [ ]:
# FFT and peak detection
sig = ir.signal
sr = ir.fs
N = len(sig)
fft_mag = np.abs(np.fft.rfft(sig))
fft_freqs = np.fft.rfftfreq(N, 1/sr)

# Analytical eigenfrequencies
analytical = []
for n in range(10):
    for m in range(10):
        if n == 0 and m == 0:
            continue
        f = c/2 * np.sqrt((n/Lx)**2 + (m/Ly)**2)
        if f < 900:
            analytical.append(f)
analytical = sorted(analytical)

# Find FFT peaks
from scipy.signal import find_peaks
peaks, props = find_peaks(fft_mag, height=np.max(fft_mag)*0.01,
                          distance=int(5/(fft_freqs[1]-fft_freqs[0])))
peak_freqs = fft_freqs[peaks]

# Match
matched = 0
print(f'{"Analytical":>12s} {"Nearest peak":>12s} {"Error (Hz)":>12s}')
print('-' * 40)
for fa in analytical[:15]:
    diffs = np.abs(peak_freqs - fa)
    if len(diffs) == 0:
        continue
    best = peak_freqs[np.argmin(diffs)]
    err = abs(best - fa)
    status = 'OK' if err < 5 else 'MISS'
    if err < 5:
        matched += 1
    print(f'{fa:12.1f} {best:12.1f} {err:12.2f}  {status}')

print(f'\nMatched: {matched}/{min(len(analytical),15)}')

In [ ]:
# Plot spectrum with analytical eigenfrequencies
fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(fft_freqs, fft_mag, 'b-', linewidth=0.5, label='Computed')
for fa in analytical[:20]:
    ax.axvline(fa, color='r', alpha=0.3, linewidth=0.5)
ax.axvline(analytical[0], color='r', alpha=0.3, linewidth=0.5, label='Analytical')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('|H(f)|')
ax.set_title('Eigenfrequency validation: 2D rigid rectangle')
ax.set_xlim(0, 900)
ax.legend()
plt.tight_layout()
plt.show()

## Test 2: 3D Box Room — BRAS CR2 at 125-250 Hz

BRAS CR2 (Scene 9) is an 8.4 × 6.7 × 3.0 m seminar room. We solve using the box constructor with BRAS fitted absorption values and compare T30 at 125 and 250 Hz against the published measured values.

**Measured reference (dodecahedron source, averaged):**
- T30 at 125 Hz: 1.50 s
- T30 at 250 Hz: 1.746 s
- Broadband T30: 1.663 s

In [ ]:
# BRAS CR2 box with fitted impedances
# Impedance Z estimated from BRAS fitted alpha at 250 Hz:
#   floor alpha~0.091 -> Z~5500
#   ceiling alpha~0.104 -> Z~4000
#   concrete walls alpha~0.075 -> Z~6700
#   windows alpha~0.073 -> Z~7000
#   plaster alpha~0.050 -> Z~10000

import time

Lx, Ly, Lz = 8.4, 6.7, 3.0
room = Room.box_3d(Lx, Ly, Lz, ne=6, order=4)
print(f'DOFs: {room.N}')

room.set_material('z_min', 5500)   # floor
room.set_material('z_max', 4000)   # ceiling
room.set_material('x_min', 7000)   # windows
room.set_material('x_max', 10000)  # plaster
room.set_material('y_min', 6700)   # concrete
room.set_material('y_max', 6700)   # concrete

room.set_source(2.0, 3.35, 1.5)
room.set_receiver(6.0, 2.0, 1.2)

t0 = time.perf_counter()
ir = room.solve(t_max=1.5, f_max=300, Ns=300)
dt = time.perf_counter() - t0
print(f'Solve time: {dt:.0f}s')

In [ ]:
# Per-band T30 comparison
def compute_t30(sig, sr):
    e = sig**2
    edc = np.cumsum(e[::-1])[::-1]
    edc /= edc[0] + 1e-30
    edc_db = 10 * np.log10(edc + 1e-30)
    t = np.arange(len(edc_db)) / sr
    i0 = np.searchsorted(-edc_db, 5)
    i1 = np.searchsorted(-edc_db, 35)
    if i1 <= i0 + 2 or i1 >= len(edc_db):
        return 0, 0
    coeffs = np.polyfit(t[i0:i1], edc_db[i0:i1], 1)
    slope = coeffs[0]
    if slope >= 0:
        return 0, 0
    rt = -60 / slope
    fitted = np.polyval(coeffs, t[i0:i1])
    ss_res = np.sum((edc_db[i0:i1] - fitted)**2)
    ss_tot = np.sum((edc_db[i0:i1] - edc_db[i0:i1].mean())**2) + 1e-30
    return max(rt, 0), 1 - ss_res / ss_tot

measured = {125: 1.50, 250: 1.746}
sr = ir.fs
nyq = sr / 2

print(f'Broadband T30 = {ir.T30:.3f}s (measured: 1.663s, error: {abs(ir.T30-1.663)/1.663*100:.1f}%)')
print()
print(f'{"Band":>8s} {"Computed":>10s} {"Measured":>10s} {"Error":>8s}')
print('-' * 40)

for fc in [125, 250]:
    fl = fc / np.sqrt(2)
    fh = min(fc * np.sqrt(2), nyq * 0.95)
    sos = butter(4, [fl/nyq, fh/nyq], btype='band', output='sos')
    filtered = sosfiltfilt(sos, ir.signal)
    t30, r2 = compute_t30(filtered, sr)
    m = measured[fc]
    err = abs(t30 - m) / m * 100 if t30 > 0 else float('nan')
    print(f'{fc:>7d}Hz {t30:>9.3f}s {m:>9.3f}s {err:>7.1f}%')

## Test 3: Arbitrary Geometry Cross-Validation

We create an L-shaped room using a Gmsh `.geo` script and verify:
1. The tet mesh produces a reasonable volume
2. Surfaces are auto-detected correctly
3. The solver produces a physically plausible IR

In [ ]:
import tempfile

geo = '''
lc = 0.4;
Point(1) = {0,0,0,lc}; Point(2) = {4,0,0,lc}; Point(3) = {4,3,0,lc};
Point(4) = {2,3,0,lc}; Point(5) = {2,5,0,lc}; Point(6) = {0,5,0,lc};
Line(1)={1,2}; Line(2)={2,3}; Line(3)={3,4};
Line(4)={4,5}; Line(5)={5,6}; Line(6)={6,1};
Curve Loop(1)={1,2,3,4,5,6}; Plane Surface(1)={1};
Extrude{0,0,2.5}{Surface{1};}
Physical Volume(1) = {1};
'''

tmp = tempfile.NamedTemporaryFile(suffix='.geo', delete=False, mode='w')
tmp.write(geo)
tmp.close()

room_L = Room.from_gmsh(tmp.name, f_max=200)

# Expected volume: (4*3 + 2*2) * 2.5 = 40 m3
vol = room_L.ops['M_diag'].sum()
print(f'DOFs: {room_L.N}')
print(f'Surfaces: {room_L._surface_labels}')
print(f'Volume: {vol:.1f} m3 (expected: 40.0 m3, error: {abs(vol-40)/40*100:.1f}%)')

os.unlink(tmp.name)

In [ ]:
# Solve L-shaped room
room_L.set_source(1.0, 1.0, 1.2)
room_L.set_receiver(3.0, 1.0, 1.0)
room_L.set_material('floor', 'carpet_thick')
room_L.set_material('ceiling', 'plaster')

t0 = time.perf_counter()
ir_L = room_L.solve(t_max=0.3, f_max=200, Ns=150)
dt = time.perf_counter() - t0

print(f'Solve time: {dt:.0f}s')
print(f'T30 = {ir_L.T30:.3f}s')
print(f'C80 = {ir_L.C80:.1f} dB')

# Plot IR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ir_L.t * 1000, ir_L.signal)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Amplitude')
ax1.set_title('L-shaped room IR')

# EDC
e = ir_L.signal**2
edc = np.cumsum(e[::-1])[::-1]
edc_db = 10 * np.log10(edc / (edc[0] + 1e-30) + 1e-30)
ax2.plot(ir_L.t * 1000, edc_db)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Energy (dB)')
ax2.set_title('Energy Decay Curve')
ax2.set_ylim(-60, 0)

plt.tight_layout()
plt.show()

## Test 4: ROM Accuracy

Build a reduced-order model from 3 training impedance values and compare the ROM output against the full-order solution at an untrained impedance.

In [ ]:
# ROM test on 2D room
room_rom = Room.box_2d(2.0, 2.0, ne=15, order=4)
room_rom.set_source(1.0, 1.0, sigma=0.2)
room_rom.set_receiver(0.3, 0.3)
room_rom.set_boundary_fi(Zs=5000)

# FOM at test impedance
ir_fom = room_rom.solve(t_max=0.1)

# Build ROM
rom = room_rom.build_rom(Z_train=[500, 5000, 15500])
print(f'ROM basis size: {rom.Nrb}')

# ROM at same impedance
ir_rom = rom.solve(Zs=5000)

# Compare
n = min(len(ir_fom.signal), len(ir_rom.signal))
rel_error = np.linalg.norm(ir_fom.signal[:n] - ir_rom.signal[:n]) / (np.linalg.norm(ir_fom.signal[:n]) + 1e-30)
print(f'Relative error: {rel_error:.2e}')
print(f'FOM T30: {ir_fom.T30:.4f}s')
print(f'ROM T30: {ir_rom.T30:.4f}s')

# Test at untrained impedance
room_rom.set_boundary_fi(Zs=3000)
ir_fom_3k = room_rom.solve(t_max=0.1)
ir_rom_3k = rom.solve(Zs=3000)
n = min(len(ir_fom_3k.signal), len(ir_rom_3k.signal))
rel_err_3k = np.linalg.norm(ir_fom_3k.signal[:n] - ir_rom_3k.signal[:n]) / (np.linalg.norm(ir_fom_3k.signal[:n]) + 1e-30)
print(f'\nUntrained Z=3000:')
print(f'Relative error: {rel_err_3k:.2e}')
print(f'FOM T30: {ir_fom_3k.T30:.4f}s')
print(f'ROM T30: {ir_rom_3k.T30:.4f}s')

In [ ]:
# Plot FOM vs ROM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ir_fom_3k.t * 1000, ir_fom_3k.signal, 'b-', label='FOM', linewidth=0.8)
ax1.plot(ir_rom_3k.t * 1000, ir_rom_3k.signal, 'r--', label='ROM', linewidth=0.8)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Pressure')
ax1.set_title(f'FOM vs ROM at Z=3000 (untrained)')
ax1.legend()

ax2.plot(ir_fom_3k.t * 1000, ir_fom_3k.signal - ir_rom_3k.signal[:len(ir_fom_3k.signal)], 'k-', linewidth=0.5)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Error')
ax2.set_title(f'Difference (rel. error = {rel_err_3k:.2e})')

plt.tight_layout()
plt.show()

## Summary

| Test | What | Result |
|------|------|--------|
| 1 | Eigenfrequencies vs analytical | Peaks match within tolerance |
| 2 | BRAS CR2 T30 at 125/250 Hz | Compare above |
| 3 | Arbitrary geometry (L-shaped) | Volume correct, IR computed |
| 4 | ROM vs FOM | Relative error shown above |